Importuojamos bibliotekos ir nustatomi failų keliai.


In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import matplotlib.dates as mdates

DATA_DIR = Path(r"...")
DATA_FILE = DATA_DIR / "main_lentele_50_su_features.parquet"

ARIMA_DIR = DATA_DIR / "arima_sarimax_46_rolling_outputsv2"
LSTM_DIR = DATA_DIR / "lstm_results_3"
LGBM_DIR = DATA_DIR / "lightgbm_results"

OUTPUT_DIR = DATA_DIR / "model_comparison_results"

Įkeliamos visų modelių prognozės.


In [ ]:
arima_pred = pd.read_parquet(
    ARIMA_DIR / "sarima_sarimax_46_rolling_forecasts.parquet"
)

lstm_pred = pd.read_parquet(
    LSTM_DIR / "lstm_test_predictions_46.parquet"
)

lgbm_pred = pd.read_parquet(
    LGBM_DIR / "lightgbm_test_predictions_46.parquet"
)

Sukuriama 7 dienų sezoninė naiviojo modelio prognozė.


In [ ]:
sales_history = pd.read_parquet(DATA_FILE)
sales_history["ds"] = pd.to_datetime(sales_history["DATE"])

if "unique_id" not in sales_history.columns:
    sales_history["unique_id"] = (
        sales_history["STORE_ID"].astype(str)
        + "_"
        + sales_history["SKU_GROUP"].astype(str)
        + "_"
        + sales_history["SKU_ID"].astype(str)
    )

sales_history["y"] = pd.to_numeric(sales_history["SALES_QTY"], errors="coerce").fillna(0)

daily_actuals = (
    sales_history
    .groupby(["unique_id", "ds"], as_index=False)["y"]
    .sum()
)

seasonal_naive_lookup = daily_actuals.rename(columns={"y": "seasonal_naive_forecast"})
seasonal_naive_lookup["ds"] = seasonal_naive_lookup["ds"] + pd.Timedelta(days=7)

seasonal_meta_cols = [
    col for col in ["STORE_ID", "SKU_GROUP", "SKU_ID", "series_type_2d"]
    if col in arima_pred.columns
]

seasonal_index = (
    arima_pred[["unique_id", "ds", "actual"] + seasonal_meta_cols]
    .rename(columns={"actual": "y"})
    .drop_duplicates(subset=["unique_id", "ds"])
    .copy()
)
seasonal_index["ds"] = pd.to_datetime(seasonal_index["ds"])

seasonal_naive_pred = seasonal_index.merge(
    seasonal_naive_lookup[["unique_id", "ds", "seasonal_naive_forecast"]],
    on=["unique_id", "ds"],
    how="left",
)


Paruošiamos SARIMA ir SARIMAX prognozės bendrai lentelei.


In [ ]:
arima_parts = []

tmp = seasonal_naive_pred.copy()
tmp = tmp.rename(columns={"seasonal_naive_forecast": "y_hat"})
tmp["model"] = "SeasonalNaive"
arima_parts.append(tmp)

tmp = arima_pred.copy()
tmp = tmp.rename(columns={
    "actual": "y",
    "sarima_forecast_raw": "y_hat"
})
tmp["model"] = "SARIMA"
arima_parts.append(tmp)

tmp = arima_pred.copy()
tmp = tmp.rename(columns={
    "actual": "y",
    "sarimax_forecast_raw": "y_hat"
})
tmp["model"] = "SARIMAX"
arima_parts.append(tmp)

arima_long = pd.concat(arima_parts, ignore_index=True)

arima_keep = [
    "unique_id",
    "STORE_ID",
    "SKU_GROUP",
    "SKU_ID",
    "series_type_2d",
    "window",
    "ds",
    "model",
    "y",
    "y_hat"
]

arima_long = arima_long[
    [col for col in arima_keep if col in arima_long.columns]
].copy()

Paruošiamos LSTM prognozės bendrai lentelei.


In [ ]:
lstm_long = lstm_pred.copy()

lstm_long = lstm_long.rename(columns={
    "LSTM": "y_hat"
})

lstm_long["model"] = "LSTM"

lstm_keep = [
    "unique_id",
    "STORE_ID",
    "SKU_GROUP",
    "SKU_ID",
    "series_type_2d",
    "sales_level",
    "zero_level",
    "mean_sales",
    "zero_pct",
    "cutoff",
    "ds",
    "model",
    "y",
    "y_hat"
]

lstm_long = lstm_long[
    [col for col in lstm_keep if col in lstm_long.columns]
].copy()

Paruošiamos LightGBM prognozės bendrai lentelei.


In [ ]:
lgbm_long = lgbm_pred.copy()

lgbm_long = lgbm_long.rename(columns={
    "LightGBM": "y_hat"
})

lgbm_long["model"] = "LightGBM"

lgbm_keep = [
    "unique_id",
    "STORE_ID",
    "SKU_GROUP",
    "SKU_ID",
    "series_type_2d",
    "sales_level",
    "zero_level",
    "mean_sales",
    "zero_pct",
    "cutoff",
    "ds",
    "model",
    "y",
    "y_hat"
]

lgbm_long = lgbm_long[
    [col for col in lgbm_keep if col in lgbm_long.columns]
].copy()

Sujungiamos visų modelių prognozės.


In [ ]:
all_preds = pd.concat(
    [arima_long, lstm_long, lgbm_long],
    ignore_index=True
)

all_preds["ds"] = pd.to_datetime(all_preds["ds"])
all_preds["y"] = all_preds["y"].astype(float)
all_preds["y_hat"] = all_preds["y_hat"].astype(float)

Apskaičiuojami MASE vardikliai ir paklaidos.


In [ ]:
seasonal_naive_errors = seasonal_naive_pred.copy()
seasonal_naive_errors["sn_abs_error"] = np.abs(
    seasonal_naive_errors["y"] - seasonal_naive_errors["seasonal_naive_forecast"]
)

mase_denominator = (
    seasonal_naive_errors
    .groupby("unique_id")
    .agg(mase_denom=("sn_abs_error", "mean"))
    .reset_index()
)

all_preds = all_preds.drop(
    columns=["mase_denom", "abs_error", "scaled_abs_error"],
    errors="ignore"
)

all_preds = all_preds.merge(
    mase_denominator,
    on="unique_id",
    how="left"
)

all_preds["error"] = all_preds["y_hat"] - all_preds["y"]
all_preds["abs_error"] = np.abs(all_preds["error"])
all_preds["squared_error"] = all_preds["error"] ** 2

all_preds["scaled_abs_error"] = (
    all_preds["abs_error"] / all_preds["mase_denom"]
)

all_preds.loc[
    ~np.isfinite(all_preds["scaled_abs_error"]),
    "scaled_abs_error"
] = np.nan

def wape_from_errors(df):
    denom = np.sum(np.abs(df["y"]))

    if denom == 0:
        return np.nan

    return np.sum(df["abs_error"]) / denom


def rmse_from_errors(df):
    return np.sqrt(np.mean(df["squared_error"]))


def pbias_from_errors(df):
    denom = np.sum(df["y"])

    if denom == 0:
        return np.nan

    return np.sum(df["error"]) / denom


def mase_from_scaled_errors(df):
    return df["scaled_abs_error"].mean()

def calculate_metrics(df):
    return pd.Series({
        "WAPE": wape_from_errors(df),
        "RMSE": rmse_from_errors(df),
        "MASE": mase_from_scaled_errors(df),
        "PBIAS": pbias_from_errors(df),
        "n_obs": len(df),
        "n_series": df["unique_id"].nunique()
    })

Suskaičiuojamos bendros ir grupinės metrikos.


In [ ]:
MODEL_ORDER = ["SARIMA", "SARIMAX", "LightGBM", "LSTM"]

all_preds = all_preds[all_preds["model"].isin(MODEL_ORDER)].copy()

if not {"STORE_ID", "SKU_GROUP", "SKU_ID"}.issubset(all_preds.columns):
    uid_parts = all_preds["unique_id"].astype(str).str.split("_", expand=True)
    all_preds["STORE_ID"] = uid_parts[0]
    all_preds["SKU_GROUP"] = uid_parts[1]
    all_preds["SKU_ID"] = uid_parts[2]

seasonal_naive_errors = seasonal_naive_pred.copy()
seasonal_naive_errors["sn_abs_error"] = np.abs(
    seasonal_naive_errors["y"] - seasonal_naive_errors["seasonal_naive_forecast"]
)

mase_denominator = (
    seasonal_naive_errors
    .groupby("unique_id")
    .agg(mase_denom=("sn_abs_error", "mean"))
    .reset_index()
)

all_preds = all_preds.drop(
    columns=["mase_denom", "error", "abs_error", "squared_error", "scaled_abs_error"],
    errors="ignore"
)

all_preds = all_preds.merge(
    mase_denominator,
    on="unique_id",
    how="left"
)

all_preds["error"] = all_preds["y_hat"] - all_preds["y"]
all_preds["abs_error"] = np.abs(all_preds["error"])
all_preds["squared_error"] = all_preds["error"] ** 2
all_preds["scaled_abs_error"] = all_preds["abs_error"] / all_preds["mase_denom"]

all_preds.loc[
    ~np.isfinite(all_preds["scaled_abs_error"]),
    "scaled_abs_error"
] = np.nan

def calc_global_metrics(df):
    y_sum = df["y"].sum()
    abs_y_sum = np.abs(df["y"]).sum()

    return pd.Series({
        "WAPE": df["abs_error"].sum() / abs_y_sum if abs_y_sum != 0 else np.nan,
        "RMSE": np.sqrt(df["squared_error"].mean()),
        "MASE": df["scaled_abs_error"].mean(),
        "PBIAS": df["error"].sum() / y_sum if y_sum != 0 else np.nan
    })

overall_4x4 = (
    all_preds
    .groupby("model")
    .apply(calc_global_metrics)
    .reindex(MODEL_ORDER)
)

display(overall_4x4)

def grouped_metric_matrix(group_col):
    result = (
        all_preds
        .groupby([group_col, "model"])
        .apply(calc_global_metrics)
        .reset_index()
        .sort_values([group_col, "model"])
    )
    return result

metrics_by_series_type = grouped_metric_matrix("series_type_2d")
display(metrics_by_series_type)

metrics_by_store = grouped_metric_matrix("STORE_ID")
display(metrics_by_store)

metrics_by_category = grouped_metric_matrix("SKU_GROUP")
display(metrics_by_category)

Apibrėžiama agreguotų prognozių braižymo funkcija.


In [ ]:
def plot_aggregated_forecasts(
    group_col=None,
    group_value=None,
    daily_start=None,
    daily_weeks=13,
    weekly_title=None,
    daily_title=None
):

    tmp = all_preds.copy()

    if group_col is not None and group_value is not None:
        if group_col not in tmp.columns:
            raise ValueError(f"Column not found: {group_col}")

        group_mask = tmp[group_col].astype(str) == str(group_value)
        tmp = tmp[group_mask].copy()

        if tmp.empty:
            available_values = sorted(all_preds[group_col].dropna().astype(str).unique())
            raise ValueError(
                f"No rows found for {group_col}={group_value!r}. "
                f"Available values: {available_values}"
            )

    tmp["ds"] = pd.to_datetime(tmp["ds"])

    if weekly_title is None:
        weekly_title = "Faktiniai pardavimai ir modelių prognozės savaitės lygmenyje"

    if daily_title is None:
        daily_title = f"Faktiniai pardavimai ir modelių prognozės dienos lygmenyje ({daily_weeks} savaičių)"

    weekly_start = pd.Timestamp("2025-07-03")
    default_daily_start = pd.Timestamp("2025-07-03")

    tmp["week"] = tmp["ds"].dt.to_period("W").dt.start_time

    weekly_actual = (
        tmp[["week", "unique_id", "y"]]
        .drop_duplicates()
        .groupby("week", as_index=False)["y"]
        .sum()
    )

    weekly_forecast = (
        tmp
        .groupby(["week", "model"], as_index=False)["y_hat"]
        .sum()
    )

    weekly_end = max(weekly_actual["week"].max(), weekly_forecast["week"].max())

    plt.figure(figsize=(10, 6))

    plt.plot(
        weekly_actual["week"],
        weekly_actual["y"],
        label="Faktiniai pardavimai",
        linewidth=3,
        color="#737373"
    )

    for model in MODEL_ORDER:
        model_df = weekly_forecast[weekly_forecast["model"] == model]

        plt.plot(
            model_df["week"],
            model_df["y_hat"],
            label=model,
            linewidth=2
        )

    plt.title(weekly_title, fontsize=16)
    plt.xlabel("Data", fontsize=14)
    ax = plt.gca()
    ax.set_xlim(left=weekly_start, right=weekly_end)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    ax.set_xticks(pd.date_range(weekly_start, weekly_end, freq="21D"))
    plt.ylabel("Pardavimai (prekės vnt.)", fontsize=14)
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

    if daily_start is None:
        daily_start = default_daily_start
    else:
        daily_start = pd.to_datetime(daily_start)

    daily_end = daily_start + pd.Timedelta(weeks=daily_weeks)

    tmp_daily = tmp[
        (tmp["ds"] >= daily_start) &
        (tmp["ds"] < daily_end)
    ].copy()

    daily_actual = (
        tmp_daily[["ds", "unique_id", "y"]]
        .drop_duplicates()
        .groupby("ds", as_index=False)["y"]
        .sum()
    )

    daily_forecast = (
        tmp_daily
        .groupby(["ds", "model"], as_index=False)["y_hat"]
        .sum()
    )

    daily_plot_end = max(daily_actual["ds"].max(), daily_forecast["ds"].max())

    plt.figure(figsize=(10, 6))

    plt.plot(
        daily_actual["ds"],
        daily_actual["y"],
        label="Faktiniai pardavimai",
        linewidth=3,
        color="#737373"
    )

    for model in MODEL_ORDER:
        model_df = daily_forecast[daily_forecast["model"] == model]

        plt.plot(
            model_df["ds"],
            model_df["y_hat"],
            label=model,
            linewidth=2
        )

    plt.title(daily_title, fontsize=16)
    plt.xlabel("Data", fontsize=14)
    ax = plt.gca()
    ax.set_xlim(left=daily_start, right=daily_plot_end)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    ax.set_xticks(pd.date_range(daily_start, daily_plot_end, freq="14D"))
    plt.ylabel("Pardavimai (prekės vnt.)", fontsize=14)
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

Braižomas bendras visų laiko eilučių grafikas.


In [ ]:
plot_aggregated_forecasts(
    weekly_title="Faktiniai pardavimai ir modelių prognozės savaitės lygmenyje",
    daily_title="Faktiniai pardavimai ir modelių prognozės dienos lygmenyje (13 savaičių)"
)

Braižomi grafikai pagal produktų grupes.


In [ ]:
plot_aggregated_forecasts(
    group_col="SKU_GROUP",
    group_value="1",
    weekly_title="Faktiniai pardavimai ir modelių prognozės savaitės lygmenyje: pirma produktų grupė",
    daily_title="Faktiniai pardavimai ir modelių prognozės dienos lygmenyje (13 savaičių): pirma produktų grupė"
)
plot_aggregated_forecasts(
    group_col="SKU_GROUP",
    group_value="2",
    weekly_title="Faktiniai pardavimai ir modelių prognozės savaitės lygmenyje: antra produktų grupė",
    daily_title="Faktiniai pardavimai ir modelių prognozės dienos lygmenyje (13 savaičių): antra produktų grupė"
)
plot_aggregated_forecasts(
    group_col="SKU_GROUP",
    group_value="3",
    weekly_title="Faktiniai pardavimai ir modelių prognozės savaitės lygmenyje: trečia produktų grupė",
    daily_title="Faktiniai pardavimai ir modelių prognozės dienos lygmenyje (13 savaičių): trečia produktų grupė"
)
plot_aggregated_forecasts(
    group_col="SKU_GROUP",
    group_value="4",
    weekly_title="Faktiniai pardavimai ir modelių prognozės savaitės lygmenyje: ketvirta produktų grupė",
    daily_title="Faktiniai pardavimai ir modelių prognozės dienos lygmenyje (13 savaičių): ketvirta produktų grupė"
)

Braižomi grafikai pagal parduotuves.


In [ ]:
plot_aggregated_forecasts(
    group_col="STORE_ID",
    group_value="A",
    weekly_title="Faktiniai pardavimai ir modelių prognozės savaitės lygmenyje: parduotuvė A",
    daily_title="Faktiniai pardavimai ir modelių prognozės dienos lygmenyje (13 savaičių): parduotuvė A"
)
plot_aggregated_forecasts(
    group_col="STORE_ID",
    group_value="B",
    weekly_title="Faktiniai pardavimai ir modelių prognozės savaitės lygmenyje: parduotuvė B",
    daily_title="Faktiniai pardavimai ir modelių prognozės dienos lygmenyje (13 savaičių): parduotuvė B"
)
plot_aggregated_forecasts(
    group_col="STORE_ID",
    group_value="C",
    weekly_title="Faktiniai pardavimai ir modelių prognozės savaitės lygmenyje: parduotuvė C",
    daily_title="Faktiniai pardavimai ir modelių prognozės dienos lygmenyje (13 savaičių): parduotuvė C"
)
plot_aggregated_forecasts(
    group_col="STORE_ID",
    group_value="D",
    weekly_title="Faktiniai pardavimai ir modelių prognozės savaitės lygmenyje: parduotuvė D",
    daily_title="Faktiniai pardavimai ir modelių prognozės dienos lygmenyje (13 savaičių): parduotuvė D"
)

Braižomi grafikai pagal laiko eilučių tipus.


In [ ]:
plot_aggregated_forecasts(
    group_col="series_type_2d",
    group_value="high_sales_low_zero_pct",
    weekly_title="Faktiniai pardavimai ir modelių prognozės savaitės lygmenyje: didelės paklausos prekės",
    daily_title="Faktiniai pardavimai ir modelių prognozės dienos lygmenyje (13 savaičių): didelės paklausos prekės"
)
plot_aggregated_forecasts(
    group_col="series_type_2d",
    group_value="medium_sales_medium_zero_pct",
    weekly_title="Faktiniai pardavimai ir modelių prognozės savaitės lygmenyje: vidutinės paklausos prekės",
    daily_title="Faktiniai pardavimai ir modelių prognozės dienos lygmenyje (13 savaičių): vidutinės paklausos prekės"
)
plot_aggregated_forecasts(
    group_col="series_type_2d",
    group_value="low_sales_high_zero_pct",
    weekly_title="Faktiniai pardavimai ir modelių prognozės savaitės lygmenyje: žemos paklausos prekės",
    daily_title="Faktiniai pardavimai ir modelių prognozės dienos lygmenyje (13 savaičių): žemos paklausos prekės"
)